# w9_scale.ipynb — ANCHOR-CAP SCALING SHOWDOWN (ace vs i2ce vs ai2ce)

THE deciding experiment for the scaling thesis: if plain anchored CE (ace)
rises with cap like i2ce does, the "I-CE scales better" claim dies; if ace
saturates/declines while i2ce climbs, it's confirmed. ai2ce rides along to
test whether the anchor rope starts paying at large caps (at 512 it is
worthless: +0.005 ZS). Grid = {ce, i2ce, ai2ce} x {512, 1024, 2048, 4096};
512 all done, i2ce@2048/@4096 done (refs), ce@2048 RESUMES from its ep250
bundle (morning pod died 11:03), everything else fresh. 1024 is a brand-new
cap: the worker builds wscan_gal_rev_g1024.npz on first run.
2048/4096 cells are big-class: run this on an A100 pod (80G for 4096).
Readout is ZS-primary (user protocol), head m4 secondary. AUTO-STOPS.

GPU packing: cap<=2048 cells run TWO per GPU (each tower uses < half
of 80G; mmap page cache is shared). 4096 runs solo. OOM under packing
auto-retries solo in wave 2.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# the scaling grid (arm, cap) -- done cells skip automatically
FLASH = [
    # cap 1024 (new point) -- small, fastest first
    ("wcle_ce_cetf", 1024),
    ("wcle_i2ce_icetf", 1024),
    ("wcle_ai2ce_icetf", 1024),
    # cap 2048 -- ce resumes ep250; i2ce done (skips)
    ("wcle_ce_cetf", 2048),
    ("wcle_ai2ce_icetf", 2048),
    ("wcle_i2ce_icetf", 2048),
    # cap 4096 -- i2ce done (skips)
    ("wcle_ce_cetf", 4096),
    ("wcle_ai2ce_icetf", 4096),
    ("wcle_i2ce_icetf", 4096),
    # 512 refs (all done, print in readout)
    ("wcle_ce_cetf", 512),
    ("wcle_i2ce_icetf", 512),
    ("wcle_ai2ce_icetf", 512),
]
# GPU packing ceiling, INCLUSIVE: a cap packs (two towers/GPU) iff
# cap <= MAXANCHOR. At 2048, the 2048 cells DO pack; only 4096 runs
# solo. Set 4096 to also pack 4096, 1024 to keep 2048 solo, 0 to
# disable packing. Session-local -- a packed crash demotes one rung
# ([512,1024,2048,4096]) for THIS pod only.
MAXANCHOR = 2048
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} scale cells")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# EXTEND FIRST (user decree): i2ce@512 + ai2ce@512 are still climbing at
# ep1000 (ZS noname .672/.657 rising) -> stretch both to 2000 epochs BEFORE
# the cap grid. extend_fs: dead-claim takeover, stale FT-best removal,
# worker rebuilds from the ep1000 checkpoint (fresh opt/rng). Under the
# ZS-only worker the whole 50..2000 curve gets val keys + zsbest.
import queue as _q
import threading

EXTENDS = [("wcle_i2ce_icetf", 512, False, 0, "clean", 16),
           ("wcle_ai2ce_icetf", 512, False, 0, "clean", 16)]
_jobs = _q.Queue()
for j6 in EXTENDS:
    _jobs.put(j6)

def _run(g):
    while True:
        try:
            j6 = _jobs.get_nowait()
        except _q.Empty:
            return
        J.extend_fs(j6, 2000, REPO, DATA_DIR, OUT_DIR,
                    full_pool_path=FULL_POOL_PATH, gpu=g)

# two slots per GPU, GPU-major order: extends co-reside on a 1-GPU pod,
# spread across GPUs when there are two.
_th = [threading.Thread(target=_run, args=(g,))
       for _ in range(2) for g in J.detect_gpus()]
for t in _th:
    t.start()
for t in _th:
    t.join()
print("extends done -> cap grid next")


In [ ]:
# Drain in TWO WAVES with GPU co-residency (ancient vicgrl/vicnogrl
# style; the full-pool mmap is page-cache shared so host RAM stays flat).
# PACK_MAX starts at the notebook constant MAXANCHOR and is SESSION-LOCAL:
# any packed job that crashes demotes the ceiling one rung (user: one
# crash per pod is a fine tuition; a persisted file could be poisoned by
# a RAM-OOM misjudged as VRAM-OOM and wrongly demote the GPU class
# globally). Demoted-out jobs reroute to the solo wave mid-flight.
import queue, subprocess, threading, time
from pathlib import Path

PACK = 2                               # co-resident towers per GPU
RUNGS = [0, 512, 1024, 2048, 4096]
PACK_MAX = MAXANCHOR                   # session-local runtime ceiling
_pmlock = threading.Lock()
print(f"[pack] MAXANCHOR = {MAXANCHOR} (notebook constant, session-local)")

def _demote(cap):
    global PACK_MAX
    with _pmlock:
        new = max(r for r in RUNGS if r < cap)
        if new < PACK_MAX:
            PACK_MAX = new
            print(f"[pack] packed job died at cap {cap} -> "
                  f"session maxanchor = {new}", flush=True)

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
todo = []
for arm, cap in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    todo.append((arm, cap, nm))

def worker(gpu, jobs, failbin, packed):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if packed and cap > PACK_MAX:
            # ceiling dropped mid-wave -- reroute to the solo wave
            failbin.append((arm, cap, nm))
            continue
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}" + (" (packed)" if packed else ""),
              flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True)   # we reaped it
            if packed:
                _demote(cap)
            failbin.append((arm, cap, nm))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

def wave(jobs_list, per_gpu, failbin, packed):
    if not jobs_list:
        return
    jobs = queue.Queue()
    for j in jobs_list:
        jobs.put(j)
    ths = [threading.Thread(target=worker, args=(g, jobs, failbin, packed))
           for _ in range(per_gpu) for g in J.detect_gpus()]
    for t in ths:
        t.start()
    for t in ths:
        t.join()

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
t0 = time.time()
refill, fails = [], []
wave([j for j in todo if j[1] <= PACK_MAX], PACK, refill, packed=True)
wave([j for j in todo if j[1] > PACK_MAX] + refill, 1, fails, packed=False)
stop_evt.set()
print(f"drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed; "
      f"session maxanchor ended at {PACK_MAX}")
for _, _, nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: the scaling grid, ZSbest-primary (val-selected).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ARMS = [("wcle_ce_cetf", "ce=ace"), ("wcle_i2ce_icetf", "i2ce"),
        ("wcle_ai2ce_icetf", "ai2ce")]

def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ARMS:
    for cap in (512, 1024, 2048, 4096):
        nm = f"w9_{arm}" + (f"_g{cap}" if cap != 512 else "")
        print(_row(f"{lab}@{cap}", nm))
    print()


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
